### Lặp Gauss-Seidel (Hệ x = Bx + d)


In [7]:
import numpy as np
from IPython.display import display, Math, Markdown

def _mat(M, p=5):
    """Chuyển đổi ma trận hoặc mảng thành chuỗi LaTeX bmatrix."""
    if hasattr(M[0], "__len__"):
        rows = " \\\\ ".join([" & ".join([f"{x:.{p}f}" for x in row]) for row in M])
        return f"\\begin{{bmatrix}} {rows} \\end{{bmatrix}}"
    inner = " \\\\ ".join([f"{x:.{p}f}" for x in M])
    return f"\\begin{{bmatrix}} {inner} \\end{{bmatrix}}"

def Gauss_Seidel_Bd(B_input, d_input, max_iter=200, epsilon=1e-5, x0=None):
    B = np.array(B_input, dtype=float)
    d = np.array(d_input, dtype=float).flatten()
    n = len(d)
    
    if x0 is None:
        X_k = np.zeros(n)
    else:
        X_k = np.array(x0, dtype=float).flatten()
        
    display(Markdown("## ❖ PHƯƠNG PHÁP LẶP GAUSS-SEIDEL (HỆ $x = Bx + d$)"))
    
    # -----------------------------------------------------------------
    # 1. KIỂM TRA ĐIỀU KIỆN HỘI TỤ & TÍNH EPSILON DỪNG
    # -----------------------------------------------------------------
    display(Markdown("### 1. Kiểm tra điều kiện hội tụ"))
    norm_B = np.linalg.norm(B, np.inf)
    if norm_B < 1:
        display(Markdown(f"Chuẩn vô cùng của ma trận $B$: $||B||_\\infty = {norm_B:.4f} < 1$, phương pháp chắc chắn hội tụ."))
    else:
        display(Markdown(f"**Cảnh báo:** $||B||_\\infty = {norm_B:.4f} \\ge 1$, phương pháp chưa chắc hội tụ."))
    
    if epsilon is not None:
        eps0 = epsilon * (1 - norm_B) / norm_B if norm_B < 1 and norm_B > 0 else epsilon
        if norm_B < 1:
            display(Math(f"\\varepsilon_0 = \\frac{{\\varepsilon(1-q)}}{{q}} = {eps0:.5e} \\quad (\\text{{với }} q = ||B||_\\infty)"))
        else:
            display(Math(f"\\varepsilon_0 = \\varepsilon = {eps0:.5e}"))
    else:
        eps0 = None
        display(Markdown("Sai số $\\varepsilon = \\text{None}$: Thuật toán chỉ dừng theo số lần lặp tối đa (`max_iter`)."))
        
    # -----------------------------------------------------------------
    # 2. QUÁ TRÌNH LẶP TÍNH TOÁN
    # -----------------------------------------------------------------
    display(Markdown("### 2. Quá trình lặp"))
    
    history = []
    diffs = []
    k = 1
    
    while True:
        X_new = np.copy(X_k)
        for i in range(n):
            sum_val = d[i]
            for j in range(n):
                sum_val += B[i, j] * X_new[j]
            X_new[i] = sum_val
            
        diff = np.linalg.norm(X_new - X_k, np.inf)
        history.append(X_new.copy())
        diffs.append(diff)
        
        stop_by_eps = (eps0 is not None and diff < eps0)
        stop_by_iter = (max_iter is not None and k >= max_iter)
        
        if stop_by_eps or stop_by_iter or k > 200:
            break
            
        X_k = np.copy(X_new)
        k += 1
        
    # -----------------------------------------------------------------
    # 3. TRÌNH BÀY KẾT QUẢ DẠNG BẢNG HÀNG NGANG
    # -----------------------------------------------------------------
    N = len(history)
    if N <= 5:
        cols = list(range(1, N+1))
    else:
        cols = [1, 2, -1, N-1, N]
        
    # Tạo tiêu đề cột: Bước lặp | x1 | x2 | ... | Sai số
    header = "| Lần lặp | " + " | ".join([f"$x_{{{i+1}}}$" for i in range(n)]) + " | $|| x_k - x_{k-1} ||_\\infty$ |"
    sep = "|---|:" + ":|:".join(["---" for _ in range(n)]) + ":|:---:|"
    lines = [header, sep]
    
    # Tạo các hàng dữ liệu tương ứng với từng bước lặp
    for c in cols:
        if c == -1:
            # Hàng biểu diễn các bước lặp bị ẩn đi bằng dấu ba chấm
            row_dots = ["..."] + ["..."] * n + ["..."]
            lines.append("| " + " | ".join(row_dots) + " |")
        else:
            row = [f"Lần {c}"]
            # Thêm giá trị của các biến x_i tại bước lặp c
            for i in range(n):
                row.append(f"{history[c-1][i]:.5f}")
            
            # Thêm giá trị sai số và ký hiệu so sánh nếu có eps0
            val_str = f"{diffs[c-1]:.5f}"
            if eps0 is not None:
                if c == N:
                    val_str += f" < \\varepsilon_0"
                elif c == N-1:
                    val_str += f" > \\varepsilon_0"
            row.append(val_str)
            lines.append("| " + " | ".join(row) + " |")
            
    display(Markdown("\n".join(lines)))
    
    # -----------------------------------------------------------------
    # 4. KẾT LUẬN NGHIỆM
    # -----------------------------------------------------------------
    display(Markdown("### 3. Kết luận"))
    display(Markdown(f"Nghiệm xấp xỉ hệ phương trình sau {N} lần lặp:"))
    display(Math(f"X \\approx {_mat(history[-1], 5)}"))

# ═══════════════════════════════════════════════════════════════════
# DỮ LIỆU ĐỀ BÀI KHỞI TẠO
# ═══════════════════════════════════════════════════════════════════
B = np.array([
    [ 0.0400,  0.0700, -0.0100,  0.0000, -0.0500],
    [-0.1000,  0.0400, -0.0200, -0.0100,  0.0400],
    [-0.0500, -0.0400,  0.0600,  0.0300,  0.0300],
    [-0.1000,  0.0900,  0.0600,  0.0400, -0.0700],
    [-0.0800, -0.1000, -0.0700,  0.0500, -0.0800]
])

d = np.array([-7.0, 6.0, -4.0, 1.0, -7.0])

# Thực thi chương trình
Gauss_Seidel_Bd(B, d, max_iter=5, epsilon=None)

## ❖ PHƯƠNG PHÁP LẶP GAUSS-SEIDEL (HỆ $x = Bx + d$)

### 1. Kiểm tra điều kiện hội tụ

Chuẩn vô cùng của ma trận $B$: $||B||_\infty = 0.3800 < 1$, phương pháp chắc chắn hội tụ.

Sai số $\varepsilon = \text{None}$: Thuật toán chỉ dừng theo số lần lặp tối đa (`max_iter`).

### 2. Quá trình lặp

| Lần lặp | $x_{1}$ | $x_{2}$ | $x_{3}$ | $x_{4}$ | $x_{5}$ | $|| x_k - x_{k-1} ||_\infty$ |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| Lần 1 | -7.00000 | 6.70000 | -3.91800 | 2.06792 | -6.73234 | 7.00000 |
| Lần 2 | -6.43520 | 6.69991 | -4.32125 | 2.54122 | -6.18704 | 0.56480 |
| Lần 3 | -6.43585 | 6.72511 | -4.31586 | 2.52463 | -6.23434 | 0.04730 |
| Lần 4 | -6.43180 | 6.72388 | -4.31761 | 2.52666 | -6.23053 | 0.00405 |
| Lần 5 | -6.43190 | 6.72401 | -4.31754 | 2.52650 | -6.23085 | 0.00032 |

### 3. Kết luận

Nghiệm xấp xỉ hệ phương trình sau 5 lần lặp:

<IPython.core.display.Math object>